In [3]:
# confirm the GPU and connect Drive

!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")
!pip install -q ultralytics

Mon Oct 19 12:56:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# unzip to Colab’s local disk

!unzip -q /content/drive/MyDrive/dental/dental_yolo.zip -d /content/dental_yolo

from pathlib import Path
root = Path("/content/dental_yolo")
(root / "data.yaml").write_text(f"""path: {root}
train: train/images
val: val/images
test: test/images
names:
  0: caries
""")
print((root / "data.yaml").read_text())

path: /content/dental_yolo
train: train/images
val: val/images
test: test/images
names:
  0: caries



In [3]:
# train

from ultralytics import YOLO

model = YOLO("yolo26n.pt")
model.train(
    data="/content/dental_yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # stop early if val mAP hasn't improved for 20 epochs
    cache=True,           # keep images in RAM for faster epochs
    project="/content/drive/MyDrive/dental/runs",   # saved to Drive, so it survives disconnects
    name="v1_yolo26n",
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dental_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c372d346190>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [6]:
# Score the model on the test set, once
from ultralytics import YOLO

best = YOLO("/content/drive/MyDrive/dental/runs/v1_yolo26n/weights/best.pt")
m = best.val(data="/content/dental_yolo/data.yaml", split="test",
             project="/content/drive/MyDrive/dental/runs", name="v1_test")

print(f"TEST  P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1888.8±1020.8 MB/s, size: 256.4 KB)
val: Scanning /content/dental_yolo/test/labels... 285 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 285/285 1.6Kit/s 0.2s
val: New cache created: /content/dental_yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.4it/s 7.6s
                   all        285        903      0.887      0.885      0.942      0.744
Speed: 2.5ms preprocess, 7.4ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/drive/MyDrive/dental/runs/v1_test-3
TEST  P=0.887  R=0.885  mAP50=0.942  mAP50-95=0.744


In [7]:
import numpy as np

conf = m.box.curves_results[3][0]          # x-axis: confidence values from 0 to 1
P = m.box.curves_results[2][1][0]          # precision at each confidence
R = m.box.curves_results[3][1][0]          # recall at each confidence

print("conf   precision  recall")
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    i = np.argmin(np.abs(conf - t))
    print(f"{t:.1f}    {P[i]:.3f}      {R[i]:.3f}")

conf   precision  recall
0.1    0.752      0.969
0.2    0.782      0.956
0.3    0.815      0.939
0.4    0.840      0.928
0.5    0.855      0.905
0.6    0.885      0.885
0.7    0.903      0.851
0.8    0.915      0.753
